In [ ]:
import numpy as np
import open3d as o3d
from PIL import Image
import matplotlib.pyplot as plt
import os

IMAGE_ORDER = [
    ("chinese_doll/1.jpeg",  960, 1280, 384, 512),
    ("chinese_doll/2.jpeg",  960, 1280, 384, 512),
    ("chinese_doll/3.jpeg",  960, 1280, 384, 512),
    ("chinese_doll/4.jpeg",  960, 1280, 384, 512),
    ("chinese_doll/5.jpeg",  960, 1280, 384, 512),
    ("chinese_doll/6.jpeg",  960, 1280, 384, 512),
    ("chinese_doll/7.jpeg",  960, 1280, 384, 512),
    ("chinese_doll/8.jpeg",  960, 1280, 384, 512),
    ("chinese_doll/9.jpeg",  960, 1280, 384, 512),
]

mesh = o3d.io.read_triangle_mesh("poisson.ply")
is_mesh = len(mesh.triangles) > 0

if is_mesh:
    print("Loaded as MESH")
    verts_world = np.asarray(mesh.vertices)
    tris = np.asarray(mesh.triangles)

    if mesh.has_vertex_colors():
        colors_v = np.asarray(mesh.vertex_colors)
    else:
        colors_v = np.ones_like(verts_world) * 0.7  

else:
    print("Loaded as POINT CLOUD")
    pcd = o3d.io.read_point_cloud("poisson.ply")
    pts_world = np.asarray(pcd.points)
    colors_pc = np.asarray(pcd.colors)

cam = np.load("cameras.npz")
poses  = cam["im_poses"].astype(np.float64)
focals = cam["im_focals"].astype(np.float64)
shapes = cam["im_shapes"].astype(np.int32)

def _score(xyz, poses_test, focals, shapes, n=5):
    s = 0
    for i in range(min(len(poses_test), n)):
        Tw = np.linalg.inv(poses_test[i])
        Pc = (Tw[:3,:3] @ xyz.T + Tw[:3,3:4]).T
        ok = Pc[:,2] > 1e-6
        Pc = Pc[ok]
        H, W = int(shapes[i,0]), int(shapes[i,1])
        f = float(np.asarray(focals[i]).flatten()[0])
        u = f * Pc[:,0]/Pc[:,2] + W/2.0
        v = f * Pc[:,1]/Pc[:,2] + H/2.0
        s += int(np.sum((u>=0)&(u<W)&(v>=0)&(v<H)))
    return s

rng = np.random.default_rng(0)
if is_mesh:
    idx = rng.choice(len(verts_world), size=min(50000, len(verts_world)), replace=False)
    xyz_sub = verts_world[idx]
else:
    idx = rng.choice(len(pts_world), size=min(50000, len(pts_world)), replace=False)
    xyz_sub = pts_world[idx]

s1 = _score(xyz_sub, poses, focals, shapes)
s2 = _score(xyz_sub, np.linalg.inv(poses), focals, shapes)
poses_c2w = poses if s1 >= s2 else np.linalg.inv(poses)

fig, axes = plt.subplots(9, 2, figsize=(14, 60))

for i, (img_path, W_orig, H_orig, W_r, H_r) in enumerate(IMAGE_ORDER):

    img_orig = np.array(Image.open(img_path).convert("RGB"))

    f_resized = float(np.asarray(focals[i]).flatten()[0])
    sx = W_orig / W_r
    sy = H_orig / H_r  

    fx = f_resized * sx
    fy = f_resized * sy
    cx = W_orig / 2.0
    cy = H_orig / 2.0

    T_w2c = np.linalg.inv(poses_c2w[i])
    R = T_w2c[:3, :3]
    t = T_w2c[:3, 3]

    rendered = np.zeros((H_orig, W_orig, 3), dtype=np.uint8)
    zbuffer = np.full((H_orig, W_orig), np.inf)

    if is_mesh:
        verts_cam = (R @ verts_world.T).T + t

        for tri in tris:
            v0, v1, v2 = verts_cam[tri]
            c0, c1, c2 = colors_v[tri]

            if (v0[2] <= 0) or (v1[2] <= 0) or (v2[2] <= 0):
                continue

            pts = np.array([v0, v1, v2])
            cols = np.array([c0, c1, c2])

            u = fx * (pts[:,0] / pts[:,2]) + cx
            v = fy * (pts[:,1] / pts[:,2]) + cy

            umin, umax = int(np.floor(u.min())), int(np.ceil(u.max()))
            vmin, vmax = int(np.floor(v.min())), int(np.ceil(v.max()))

            if umax < 0 or umin >= W_orig or vmax < 0 or vmin >= H_orig:
                continue

            umin = max(0, umin); umax = min(W_orig-1, umax)
            vmin = max(0, vmin); vmax = min(H_orig-1, vmax)

            for px in range(umin, umax+1):
                for py in range(vmin, vmax+1):
                    # barycentric
                    denom = ((v[1]-v[2])*(u[0]-u[2]) + (u[2]-u[1])*(v[0]-v[2]))
                    if abs(denom) < 1e-8:
                        continue
                    w0 = ((v[1]-v[2])*(px-u[2]) + (u[2]-u[1])*(py-v[2])) / denom
                    w1 = ((v[2]-v[0])*(px-u[2]) + (u[0]-u[2])*(py-v[2])) / denom
                    w2 = 1 - w0 - w1
                    if (w0>=0) and (w1>=0) and (w2>=0):
                        z = w0*v0[2] + w1*v1[2] + w2*v2[2]
                        if z < zbuffer[py, px]:
                            zbuffer[py, px] = z
                            color = w0*c0 + w1*c1 + w2*c2
                            rendered[py, px] = (color*255).astype(np.uint8)

        title_right = "Mesh projection"

    else:
        pts_cam = (R @ pts_world.T).T + t
        in_front = pts_cam[:, 2] > 1e-6
        pts_f = pts_cam[in_front]
        col_f = colors_pc[in_front]

        u = fx * (pts_f[:, 0] / pts_f[:, 2]) + cx
        v = fy * (pts_f[:, 1] / pts_f[:, 2]) + cy
        z = pts_f[:, 2]

        u_int = np.round(u).astype(np.int32)
        v_int = np.round(v).astype(np.int32)

        valid = (u_int >= 0) & (u_int < W_orig) & (v_int >= 0) & (v_int < H_orig)
        u_int  = u_int[valid]
        v_int  = v_int[valid]
        z_val  = z[valid]
        col_v  = col_f[valid]

        order  = np.argsort(-z_val)
        rendered[v_int[order], u_int[order]] = (col_v[order]*255).astype(np.uint8)

        title_right = "Point cloud projection"

    axes[i, 0].imshow(img_orig)
    axes[i, 0].axis("off")

    axes[i, 1].imshow(rendered)
    axes[i, 1].set_title(title_right)
    axes[i, 1].axis("off")

plt.tight_layout()
plt.show()
